In [43]:
from sklearn.datasets import fetch_20newsgroups
# 20개토픽중에 선택 ( 무신론, 종교, 컴퓨터그래픽, 우주과학)
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
newsgroups_train = fetch_20newsgroups(subset='train',remove=('headers','footers','quotes'),categories=categories)
newsgroups_test = fetch_20newsgroups(subset='test',remove=('headers','footers','quotes'),categories=categories)

In [44]:
len(newsgroups_train.data), len(newsgroups_test.data), set(newsgroups_test.target)

(2034, 1353, {np.int64(0), np.int64(1), np.int64(2), np.int64(3)})

In [45]:
print(newsgroups_train.data[100]), newsgroups_train.target[100], newsgroups_train.target_names[ newsgroups_train.target[100]  ]


This is a good point, but I think "average" people do not take up Christianity
so much out of fear or escapism, but, quite simply, as a way to improve their
social life, or to get more involved with American culture, if they are kids of
immigrants for example.  Since it is the overwhelming major religion in the
Western World (in some form or other), it is simply the choice people take if
they are bored and want to do something new with their lives, but not somethong
TOO new, or TOO out of the ordinary.  Seems a little weak, but as long as it
doesn't hurt anybody...
 

These are good quotes, and I agree with both of them, but let's make sure to
alter the scond one so that includes something like "...let him be, as long as
he is not preventing others from finding their peace." or something like that. 
(Of course, I suppose, if someone were REALLY "at peace", there would be no
need for inflicting evangelism)


Well, it is a sure thing we will have to live with them all our lives.  Their


(None, np.int64(0), 'alt.atheism')

In [46]:
x_train = newsgroups_train.data
y_train = newsgroups_train.target
x_test = newsgroups_test.data
y_test = newsgroups_test.target

In [47]:
# NLP  문자 -> 학습가능한 형태의 숫자(Vector) ->모델학습
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2000)
x_train_cv = cv.fit_transform(x_train)
x_test_cv = cv.transform(x_test)
x_train_cv.shape, x_test_cv.shape

((2034, 2000), (1353, 2000))

In [48]:
from sklearn.naive_bayes import MultinomialNB
NB_clf = MultinomialNB()
NB_clf.fit(x_train_cv, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [49]:
NB_clf.score(x_train_cv, y_train), NB_clf.score(x_test_cv, y_test)

(0.8200589970501475, 0.7317073170731707)

In [50]:
NB_clf.predict(x_test_cv[:3]), y_test[:3]

(array([2, 1, 1]), array([2, 1, 1]))

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=2000)
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)
x_train_tfidf.shape, x_test_tfidf.shape

((2034, 2000), (1353, 2000))

In [52]:
from sklearn.naive_bayes import MultinomialNB
NB_clf = MultinomialNB()
NB_clf.fit(x_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [53]:
NB_clf.score(x_train_tfidf, y_train), NB_clf.score(x_test_tfidf, y_test)

(0.8525073746312685, 0.7376201034737621)

In [55]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier, LassoCV
logistic = LogisticRegression()
ridge = RidgeClassifier()
lasso = LassoCV()

logistic.fit(x_train_tfidf, y_train)
ridge.fit(x_train_tfidf, y_train)
lasso.fit(x_train_tfidf, y_train)

print( logistic.score(x_train_tfidf, y_train), logistic.score(x_test_tfidf, y_test) )
print( ridge.score(x_train_tfidf, y_train), ridge.score(x_test_tfidf, y_test) )
print( lasso.score(x_train_tfidf, y_train), lasso.score(x_test_tfidf, y_test) )

0.9203539823008849 0.7317073170731707
0.9587020648967551 0.7427937915742794
0.48895982123641346 0.14656652054016017


In [59]:
# 규제 강조 튜닝
import numpy as np
alpha_lists = np.linspace(0.1, 10, 50)
params = {
    'alpha' : alpha_lists
}
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(RidgeClassifier(), param_grid=params)
grid.fit(x_train_tfidf, y_train)

,estimator,RidgeClassifier()
,param_grid,{'alpha': array([ 0.1 ... 10. ])}
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,np.float64(2.322448979591837)


In [62]:
best_model = grid.best_estimator_
best_model.score(x_train_tfidf, y_train), best_model.score(x_test_tfidf, y_test)

(0.9360865290068829, 0.7479674796747967)

In [65]:
news_data = {
    'content': [
        # IT/과학
        "삼성전자가 차세대 폴더블 스마트폰을 공개하며 시장 공략에 나섰습니다.",
        "인공지능 기술이 발전함에 따라 반도체 수요가 급증하고 있습니다.",
        "구글의 새로운 AI 모델이 인간과의 대화에서 자연스러운 반응을 보였습니다.",
        # 경제
        "미국 연준이 금리를 동결하면서 국내 증시는 혼조세를 보였습니다.",
        "최근 물가 상승률이 둔화되면서 하반기 경기 회복 기대감이 커지고 있습니다.",
        "대기업들의 실적 발표가 이어지는 가운데 반도체 업종이 강세를 보였습니다.",
        # 사회
        "이번 주말 서울 도심에서 대규모 집회가 예정되어 교통 혼잡이 예상됩니다.",
        "경찰은 최근 급증하는 보이스피싱 범죄를 막기 위해 집중 단속에 나섰습니다.",
        "정부는 저출산 문제 해결을 위해 새로운 육아 지원 정책을 발표했습니다.",
        # 정치
        "여야 정치권은 국회 본회의를 열고 민생 법안 처리에 합의했습니다.",
        "대통령은 국무회의에서 경제 활성화를 위한 규제 개혁을 강조했습니다.",
        "새로운 정당 창당 소식에 정치권의 판도가 요동치고 있습니다."
    ],
    'category': ['IT/과학', 'IT/과학', 'IT/과학', '경제', '경제', '경제', '사회', '사회', '사회', '정치', '정치', '정치']
}

In [66]:
from konlpy.tag import Okt
okt = Okt()

In [ ]:
# tagging이 nouns 이고 글자 길이가 2글자 이상인 단어들만 token화
[word for word in okt.nouns("삼성전자가 차세대 폴더블 스마트폰을 공개하며 시장 공략에 나섰습니다.") if len(word) >= 2]

['전자', '차세대', '더블', '스마트폰', '공개', '시장', '공략']

In [68]:
# 한글 토크나이저
def korean_tokenizer(text):
    okt = Okt()
    return [word for word in okt.nouns(text) if len(word) >= 2]

In [75]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
tfidf = TfidfVectorizer(tokenizer=korean_tokenizer)

x, y = news_data['content'], news_data['category']
x_tfidf = tfidf.fit_transform(x)
x_tfidf.shape

x_train, x_test, y_train, y_test = train_test_split(x_tfidf,y,random_state=42)

c:\Users\Playdata\miniconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [76]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB(alpha=0.1)
nb.fit(x_train,y_train)
nb.score(x_train,y_train), nb.score(x_test,y_test)

(1.0, 0.6666666666666666)

In [77]:
y_train, y_test
nb.predict(x_test), y_test

(array(['IT/과학', '정치', 'IT/과학'], dtype='<U5'), ['정치', '정치', 'IT/과학'])